# CS-781 Phase 2 (L-NER) — `p2_12_moe_hc_full_ft_bilstm_clsw`

**Phase-1 model:** MoE (Muon + final LayerNorm), continuation 2 on high-court data — Phase-1 Codabench 0.8376
**Technique:** **Full fine-tuning + BiLSTM prediction head + class weights**: like p2_06, but the head is a bidirectional LSTM (256 per direction) over the model's hidden states, so every token gets RIGHT context (e.g. '..Appellant' after a name) without changing the causally pretrained layers (p2_04 showed that removing the causal mask hurts). Tests the prediction-head factor.

What the notebook does (one Run All):
1. downloads the Phase-2 data, your Phase-1 tokenizer and checkpoint from the bucket; checks the weights loaded (next-token loss on NER text);
2. adds a token-classification head (BIO tags over the 12 labels) and applies the technique;
3. trains on **fold1**, scores the **strict macro-F1 on fold2 after every epoch** and keeps the best epoch (2-fold protocol: the test fold is never used);
4. full held-out report: per-label P/R/F1, boundary vs label errors, label confusions, the official `evaluate_phase2.py` on the same predictions, trainable parameters, peak GPU memory, speed;
5. retrains on fold1+fold2 for the chosen number of epochs, predicts the 35 test documents, validates the format and writes `submission_p2_12_moe_hc_full_ft_bilstm_clsw.zip` (`prediction.json` at the zip root);
6. uploads the results JSON, held-out predictions and the trained weights (only the changed parameters) to the bucket.

**Kaggle:** Accelerator *GPU T4* (one GPU is used), secrets `HF` and `HF_WRITE`. Expected time ≈ 25–50 min (not measured yet). Set `QUICK_TEST = True` first for a ~5-min end-to-end check if you like.

All settings in the next cell are **your choices**; the values there are untuned placeholders.

## 1. Settings

In [1]:
import json
# =====================================================================================================
# SETTINGS -- every value below is YOUR choice (the numbers are placeholders, not tuned)
# =====================================================================================================
RUN_NAME = "p2_12_moe_hc_full_ft_bilstm_clsw"            # names the results / weights / submission files
MODEL_KEY = "moe_hc"
PHASE1_CHECKPOINT = "phase1_decoder_only_gqa_kvcache_moe_muon_finalnorm_continued2_highcourt_fp16_full_checkpoint.pt"
TOKENIZER_NAME = "a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json"

# ---- technique ---------------------------------------------------------------------------------------
TECHNIQUE = "full_ft"         # full_ft | partial_ft | bitfit | lora | lorma | adapter | prompt | prefix
RANK = 16                     # lora / lorma rank r
ALPHA = 32                    # lora / lorma scale alpha (update is scaled by alpha / r; LoRMA_pi uses no scale)
LORMA_VARIANT = "plus"        # "plus" = LoRMA+ (additive rank inflation) | "pi" = LoRMA_pi (permutation inflation)
LORA_DROPOUT = 0.0
PEFT_TARGETS = ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.out_proj"]   # linear layers adapted by lora / lorma (name suffixes)
PARTIAL_TOP_BLOCKS = 2        # partial_ft: how many top blocks are trained
ADAPTER_TYPE = "houlsby"      # adapter: "houlsby" (after attention and FFN) | "pfeiffer" (after FFN only)
ADAPTER_DIM = 64              # adapter bottleneck width
PROMPT_LEN = 20               # prompt tuning: soft tokens prepended to every window
PREFIX_LEN = 20               # prefix tuning: learned key/value vectors per layer
MOE_BIAS_RATE_FT = 0.0       # MoE routing-bias update rate during fine-tuning (Phase 1 used 0.001). 0.0 = keep the pretrained routing fixed.

# ---- task modelling ----------------------------------------------------------------------------------
INPUT_MODE = "none"           # "none" = exact Phase-1 tokens | "punct" = split punctuation before BPE (see the step-0 notebook)
ATTENTION = "causal"          # "causal" = as pretrained (each token sees only its left) | "bidirectional" = drop the causal mask
HEAD_TYPE = "bilstm"   # "linear" | "mlp" | "bilstm" (bidirectional LSTM over the backbone states, then linear)
HEAD_LSTM_HIDDEN = 256        # bilstm head: hidden size per direction
HEAD_DROPOUT = 0.1
HEAD_NORM = False             # extra LayerNorm in the head
CLASS_WEIGHTING = "inv_sqrt"   # "none" | "inv_sqrt" | "inv" (loss weights from tag frequencies, against class imbalance)
DECODE = "viterbi"            # "viterbi" = best BIO-valid tag sequence | "argmax" = per-token argmax with lenient repair
MERGE = "context"             # overlapping windows: "context" = keep the window giving the token most context | "average"
BOUNDARY_REPAIR = True        # trim punctuation at span edges with the table learned on the training fold(s)

# ---- training -----------------------------------------------------------------------------------------
WINDOW = 2048                 # tokens per window (<= 2048; with prompt tuning <= 2048 - PROMPT_LEN)
TRAIN_STRIDE = 1024           # training windows overlap by WINDOW - TRAIN_STRIDE
INFER_STRIDE = 1024
BATCH = 2                     # windows per forward pass (lower to 1 if CUDA runs out of memory)
ACCUM = 4                     # gradient accumulation -> effective batch = BATCH * ACCUM windows
EPOCHS = 10                   # maximum; the best epoch on the held-out fold is kept
PATIENCE = 3                  # stop after this many epochs without held-out improvement (0 = never)
LR = 3e-5                     # learning rate of every trainable parameter outside the head (backbone / LoRA / LoRMA / adapters / biases / prompt / prefix)
HEAD_LR = 1e-3                # learning rate of the new head (incl. the BiLSTM head)
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
GRAD_CLIP = 1.0
AMP = "fp16"                  # "fp16" mixed precision on GPU | "none"
SEED = 0
INFER_BATCH = 4
KD_ALPHA = 0.0; KD_TEMPERATURE = 2.0     # (used only by the distillation notebook)

# ---- evaluation protocol ---------------------------------------------------------------------------------
CV_TRAIN, CV_EVAL = "fold1", "fold2"     # train on one fold, pick the epoch and report scores on the other
REFIT_ON_BOTH = True          # then retrain on fold1+fold2 for the best epoch count and predict the test set with that model
                              # (False = predict the test set with the fold1 model; saves ~half the time)
QUICK_TEST = False            # True = tiny run (2 docs per fold, 1 epoch, 512-token windows) to check the notebook end to end

C_KEYS = ["TECHNIQUE", "RANK", "ALPHA", "LORMA_VARIANT", "LORA_DROPOUT", "PEFT_TARGETS", "PARTIAL_TOP_BLOCKS", "ADAPTER_TYPE",
          "ADAPTER_DIM", "PROMPT_LEN", "PREFIX_LEN", "INPUT_MODE", "ATTENTION", "HEAD_TYPE", "HEAD_LSTM_HIDDEN", "HEAD_DROPOUT", "HEAD_NORM",
          "CLASS_WEIGHTING", "DECODE", "MERGE", "BOUNDARY_REPAIR", "WINDOW", "TRAIN_STRIDE", "INFER_STRIDE", "BATCH", "ACCUM",
          "EPOCHS", "PATIENCE", "LR", "HEAD_LR", "WEIGHT_DECAY", "WARMUP_FRAC", "GRAD_CLIP", "AMP", "SEED", "INFER_BATCH",
          "KD_ALPHA", "KD_TEMPERATURE", "CV_TRAIN", "CV_EVAL", "REFIT_ON_BOTH"]
if QUICK_TEST:
    EPOCHS, WINDOW, TRAIN_STRIDE, INFER_STRIDE, PATIENCE = 1, 512, 512, 256, 0
    RUN_NAME += "_quicktest"
C = {k: globals()[k] for k in C_KEYS}
print(json.dumps(C, indent=None, default=str))


{"TECHNIQUE": "full_ft", "RANK": 16, "ALPHA": 32, "LORMA_VARIANT": "plus", "LORA_DROPOUT": 0.0, "PEFT_TARGETS": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.out_proj"], "PARTIAL_TOP_BLOCKS": 2, "ADAPTER_TYPE": "houlsby", "ADAPTER_DIM": 64, "PROMPT_LEN": 20, "PREFIX_LEN": 20, "INPUT_MODE": "none", "ATTENTION": "causal", "HEAD_TYPE": "bilstm", "HEAD_LSTM_HIDDEN": 256, "HEAD_DROPOUT": 0.1, "HEAD_NORM": false, "CLASS_WEIGHTING": "inv_sqrt", "DECODE": "viterbi", "MERGE": "context", "BOUNDARY_REPAIR": true, "WINDOW": 2048, "TRAIN_STRIDE": 1024, "INFER_STRIDE": 1024, "BATCH": 2, "ACCUM": 4, "EPOCHS": 10, "PATIENCE": 3, "LR": 3e-05, "HEAD_LR": 0.001, "WEIGHT_DECAY": 0.01, "WARMUP_FRAC": 0.06, "GRAD_CLIP": 1.0, "AMP": "fp16", "SEED": 0, "INFER_BATCH": 4, "KD_ALPHA": 0.0, "KD_TEMPERATURE": 2.0, "CV_TRAIN": "fold1", "CV_EVAL": "fold2", "REFIT_ON_BOTH": true}


## 2. Setup: login, bucket

In [2]:
!pip install -q -U "huggingface_hub>=1.5"
import os, json, shutil, subprocess, sys, time, zipfile, random, math, gc
from collections import Counter, defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download, login, download_bucket_files, batch_bucket_files
from kaggle_secrets import UserSecretsClient

NB_START = time.time()
WORK = "/kaggle/working"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # old repo, read-only fallback
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def get_dataset_file(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)


def fetch_model_file(name, local_path=None):
    local_path = local_path or f"{WORK}/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    # fallback: the OLD model repo (the Phase-1 notebooks do the same; the tokenizer only lives there)
    p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
    shutil.copy(p, local_path)
    print(f"(found {name} in the old repo {HF_CHECKPOINT_REPO})")
    return local_path


def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    try:
        batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
        print(f"uploaded {name} to bucket {HF_BUCKET}")
    except Exception as e:
        print(f"(upload of {name} failed: {type(e).__name__}: {e}; the file is still in {WORK})")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.0/838.0 kB 11.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.1.1 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.1.1 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.1.1 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.1.1 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.1.1 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.1.1 which is incompatible.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


device: cuda Tesla T4


## 3. Phase-2 data

In [3]:
P2 = "Phase_2_LNER"
def load_json(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)

LABELS = [l.strip() for l in open(get_dataset_file(f"{P2}/Train/labels.txt"), encoding="utf-8") if l.strip()]
TEXTS = {f: load_json(get_dataset_file(f"{P2}/Train/{f}-text.json")) for f in ("fold1", "fold2")}
GOLDS = {f: load_json(get_dataset_file(f"{P2}/Train/{f}-gold.json")) for f in ("fold1", "fold2")}
TEST_TEXTS = load_json(get_dataset_file(f"{P2}/Test/fold3-text.json"))
for f in TEXTS:
    assert set(TEXTS[f]) == set(GOLDS[f])
    assert all(TEXTS[f][d][e[0]:e[1]] == e[3] for d in GOLDS[f] for e in GOLDS[f][d]), "gold offsets do not match the text"
if QUICK_TEST:
    for f in TEXTS:
        keep = list(TEXTS[f])[:2]
        TEXTS[f] = {k: TEXTS[f][k] for k in keep}; GOLDS[f] = {k: GOLDS[f][k] for k in keep}
    TEST_TEXTS = {k: TEST_TEXTS[k] for k in list(TEST_TEXTS)[:3]} | {k: v for k, v in TEST_TEXTS.items() if not v.strip()}
print("labels:", LABELS)
print({f: (len(TEXTS[f]), sum(len(v) for v in GOLDS[f].values())) for f in TEXTS}, "| test docs:", len(TEST_TEXTS))


labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
{'fold1': (35, 4120), 'fold2': (35, 4273)} | test docs: 35


## 4. Your Phase-1 BPE tokenizer (class unchanged from Phase 1), loaded from the bucket

In [4]:
import array, heapq, time
from collections import Counter, defaultdict

BPE_MIN_WORD_FREQ = 1      # 1 = learn merges from every word type (exactly the Assignment-1 algorithm)
_END = "</w>"


class BPETokenizer:
    """Byte-level BPE (Assignment 1). Initial symbols are the UTF-8 bytes of each word, held as chr(b), plus
    '</w>' marking the end of a word so merges never cross word boundaries. Ties in pair frequency break
    lexicographically. Text is NOT lowercased."""

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.merges = []
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {}
        self._ranks = {}
        self._id_cache = {}

    # ---- helpers ----
    def _word_symbols(self, word):
        syms = [chr(b) for b in word.encode("utf-8")]
        syms.append(_END)
        return syms

    def _merge(self, symbols, pair):
        res, i = [], 0
        a, b = pair
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                res.append(a + b)
                i += 2
            else:
                res.append(symbols[i])
                i += 1
        return res

    # ---- training ----
    def fit_file(self, path, block_chars=50_000_000, min_word_freq=1):
        """Count word frequencies by streaming the corpus file in blocks, then learn the merges."""
        t0 = time.time()
        word_freqs = Counter()
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()          # the last word may be cut in half: finish it with the next block
                else:
                    leftover = ""
                word_freqs.update(parts)
        if leftover:
            word_freqs[leftover] += 1
        print(f"  counted {sum(word_freqs.values()):,} words ({len(word_freqs):,} distinct) in {time.time() - t0:.0f}s", flush=True)
        return self.fit_word_freqs(word_freqs, min_word_freq)

    def fit_word_freqs(self, word_freqs, min_word_freq=1):
        t0 = time.time()
        target = self.vocab_size
        words = [w for w, c in word_freqs.items() if c >= min_word_freq]
        freqs = [word_freqs[w] for w in words]
        splits = [self._word_symbols(w) for w in words]
        del word_freqs

        base = set()
        for syms in splits:
            base.update(syms)
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.merges = []
        self.itos.extend(sorted(base))
        print(f"  base byte alphabet: {len(base)} symbols; learning merges from {len(words):,} word types", flush=True)

        # pair_counts: how often each adjacent pair occurs (weighted by word frequency).
        # pair_words: which word types contain each pair, so a merge only revisits those words.
        pair_counts = {}
        pair_words = defaultdict(set)
        for i, syms in enumerate(splits):
            f = freqs[i]
            for j in range(len(syms) - 1):
                p = (syms[j], syms[j + 1])
                pair_counts[p] = pair_counts.get(p, 0) + f
                pair_words[p].add(i)

        # Max-heap of (-count, pair): pops the most frequent pair, ties -> lexicographically smallest pair
        # (identical to Assignment 1's min over (-count, pair)). Entries go stale when a count changes; a
        # fresh entry is pushed for every changed pair and stale ones are skipped when they reach the top.
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        while len(self.itos) < target:
            best, best_f = None, 0
            while heap:
                neg_c, p = heap[0]
                if pair_counts.get(p, 0) == -neg_c:
                    best, best_f = p, -neg_c
                    heapq.heappop(heap)
                    break
                heapq.heappop(heap)                  # stale entry
            if best is None:
                break
            if best_f < 2:
                print("  stopping: no pair occurs more than once", flush=True)
                break

            self.merges.append([best[0], best[1]])
            self.itos.append(best[0] + best[1])

            changed = set()
            for i in list(pair_words[best]):
                syms, f = splits[i], freqs[i]
                for j in range(len(syms) - 1):
                    p = (syms[j], syms[j + 1])
                    c = pair_counts.get(p, 0) - f
                    if c <= 0:
                        pair_counts.pop(p, None)
                    else:
                        pair_counts[p] = c
                    pair_words[p].discard(i)
                    changed.add(p)
                new_syms = self._merge(syms, best)
                splits[i] = new_syms
                for j in range(len(new_syms) - 1):
                    p = (new_syms[j], new_syms[j + 1])
                    pair_counts[p] = pair_counts.get(p, 0) + f
                    pair_words[p].add(i)
                    changed.add(p)
            for p in changed:
                c = pair_counts.get(p)
                if c:
                    heapq.heappush(heap, (-c, p))
            del pair_words[best]

            if len(self.merges) % 1000 == 0:
                print(f"  merges {len(self.merges):,} | vocab {len(self.itos):,} | {time.time() - t0:.0f}s", flush=True)

        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self._ranks = {(a, b): r for r, (a, b) in enumerate(self.merges)}
        self._id_cache = {}
        self.vocab_size = len(self.itos)      # the size the embedding table must actually have
        return self

    # ---- save / load (so the tokenizer is trained once and then reused) ----
    def save(self, path, meta=None):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges, "itos": self.itos,
                       "meta": meta or {}}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(data["vocab_size"])
        tok.merges = [list(m) for m in data["merges"]]
        tok.itos = list(data["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok._ranks = {(a, b): r for r, (a, b) in enumerate(tok.merges)}
        tok._id_cache = {}
        tok.vocab_size = len(tok.itos)
        return tok

    # ---- inference ----
    def _tokenize_word(self, word):
        syms = self._word_symbols(word)
        while len(syms) > 1:
            best_rank, best_j = None, -1
            for j in range(len(syms) - 1):
                r = self._ranks.get((syms[j], syms[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_j = r, j
            if best_rank is None:
                break
            syms = self._merge(syms, (syms[best_j], syms[best_j + 1]))
        return syms

    def _word_ids(self, word):
        ids = self._id_cache.get(word)
        if ids is None:
            unk = self.stoi["<unk>"]
            ids = [self.stoi.get(t, unk) for t in self._tokenize_word(word)]     # symbols never seen in training -> <unk>
            self._id_cache[word] = ids
        return ids

    def tokenize(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._tokenize_word(word))
        return out

    def get_vocab_size(self):
        return len(self.itos)

    def encode(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._word_ids(word))
        return out

    def encode_file(self, path, block_chars=5_000_000):
        """Encode a whole text file into a packed array of ids, block by block (bounded memory)."""
        out = array.array("i")
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()
                else:
                    leftover = ""
                cache = self._id_cache
                for word in parts:
                    ids = cache.get(word)
                    if ids is None:
                        ids = self._word_ids(word)
                    out.extend(ids)
        if leftover:
            out.extend(self._word_ids(leftover))
        return out

    def decode(self, ids):
        """ids -> string. Words are re-joined with single spaces; <unk> decodes to '?'."""
        words, cur = [], bytearray()
        for i in ids:
            i = int(i)
            if not 0 <= i < len(self.itos):
                continue
            tok = self.itos[i]
            if tok in ("<pad>", "<s>", "</s>"):
                continue
            if tok == "<unk>":
                cur.extend(b"?")
                continue
            ends_word = tok.endswith(_END)
            if ends_word:
                tok = tok[: -len(_END)]
            cur.extend(ord(c) & 0xFF for c in tok)
            if ends_word:
                words.append(bytes(cur).decode("utf-8", errors="replace"))
                cur = bytearray()
        if cur:
            words.append(bytes(cur).decode("utf-8", errors="replace"))
        return " ".join(words)



bpe_tokenizer = BPETokenizer.load(fetch_model_file(TOKENIZER_NAME))
VOCAB = bpe_tokenizer.get_vocab_size()
print("tokenizer:", TOKENIZER_NAME, "| vocab", VOCAB)


(…)full_v40000_160000000w_seed42_minf1.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

(found a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json in the old repo kavincg23/cs781-checkpoints)
tokenizer: a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json | vocab 40000


## 5. Character offsets, BIO tags, span decoding, boundary repair, strict scorer

In [5]:
# ===== Phase 2 L-NER: character offsets for the Phase-1 BPE tokenizer, BIO tags, span decoding =====
# Ancillary pipeline code (data handling / alignment / decoding). It does NOT change the tokenizer: the token ids
# produced here are exactly BPETokenizer.encode(text) (checked on every document by check_encode_identical).

LABELS_DEFAULT = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
_END_MARK = "</w>"


def iter_words(text):
    """(start_char, end_char) of every whitespace-separated word, using exactly str.split()'s notion of whitespace
    (str.isspace), so the words are the same ones BPETokenizer.encode() sees."""
    n, i = len(text), 0
    while i < n:
        while i < n and text[i].isspace():
            i += 1
        if i >= n:
            break
        j = i
        while j < n and not text[j].isspace():
            j += 1
        yield i, j
        i = j


import re as _re
PIECE_MODES = {
    "none": None,                                                   # exactly the Phase-1 tokenization
    "punct": _re.compile(r"\w+|[^\w\s]"),                          # split every punctuation char off
    "punct_digit": _re.compile(r"[^\W\d_]+|\d+|_+|[^\w\s]"),       # also split letters from digits
}


def iter_pieces(text, mode="none"):
    """Words as the tokenizer sees them. mode 'none' = whitespace words (Phase-1 behaviour); the other modes split
    each whitespace word further, which changes the token ids versus pretraining (a design choice to measure)."""
    rx = PIECE_MODES[mode]
    for ws, we in iter_words(text):
        if rx is None:
            yield ws, we
        else:
            for m in rx.finditer(text, ws, we):
                yield m.start(), m.end()


def encode_with_offsets(tok, text, mode="none"):
    """Return (ids, spans): ids == tok.encode(text); spans[k] = (start_char, end_char) of token k in `text`
    (end exclusive). A token's symbols are UTF-8 bytes held as chr(b); when a token boundary falls inside a
    multi-byte character, that character is given to the token holding its first byte (the next token then starts
    after it; if a token holds only continuation bytes it gets an empty span at that position)."""
    unk = tok.stoi["<unk>"]
    ids, spans = [], []
    for ws, we in iter_pieces(text, mode):
        word = text[ws:we]
        syms = tok._tokenize_word(word)
        cached = tok._id_cache.get(word)
        if cached is None:
            cached = [tok.stoi.get(t, unk) for t in syms]
            tok._id_cache[word] = cached
        # byte offset -> char offset inside the word
        byte_start_of_char = []
        b = 0
        for ch in word:
            byte_start_of_char.append(b)
            b += len(ch.encode("utf-8"))
        total_bytes = b
        byte_to_char = [0] * (total_bytes + 1)
        for ci, bs in enumerate(byte_start_of_char):
            nb = (byte_start_of_char[ci + 1] if ci + 1 < len(byte_start_of_char) else total_bytes) - bs
            for k in range(nb):
                byte_to_char[bs + k] = ci if k == 0 else ci + 1      # a continuation byte "belongs" after its char start
        byte_to_char[total_bytes] = len(word)
        bpos = 0
        for sym in syms:
            core = sym[:-len(_END_MARK)] if sym.endswith(_END_MARK) else sym
            nbytes = len(core)                                       # each symbol char is one byte
            cs = byte_to_char[bpos] if bpos < total_bytes else len(word)
            bpos += nbytes
            ce = byte_to_char[bpos] if bpos < total_bytes else len(word)
            if ce < cs:
                ce = cs
            spans.append((ws + cs, ws + ce))
        ids.extend(cached)
    return ids, spans


def check_encode_identical(tok, docs):
    """Assert our ids equal tok.encode on every doc and that spans are monotone and inside the text."""
    for did, text in docs.items():
        ids, spans = encode_with_offsets(tok, text)
        ref = tok.encode(text)
        assert ids == ref, f"doc {did}: ids differ from BPETokenizer.encode"
        prev = 0
        for (s, e) in spans:
            assert prev <= s <= e <= len(text), f"doc {did}: bad span {(s, e)}"
            prev = e
    return True


# ---------------- BIO tags ----------------
IGNORE = -100        # label for tokens that cover no character (loss ignores them; decoding skips them)
def bio_tag_names(labels):
    return ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]


def gold_to_token_tags(spans, ents, labels):
    """Token tag ids (0 = O, 1+2*li = B-label, 2+2*li = I-label). A token is inside an entity if its char span
    overlaps the entity. Also returns how many entities have a start/end that is NOT on a token boundary."""
    tag = [0 if e > s else IGNORE for s, e in spans]                # empty-span tokens (e.g. a lone '</w>') -> ignored
    lab_idx = {l: i for i, l in enumerate(labels)}
    starts = {s for s, e in spans if e > s}
    ends = {e for s, e in spans if e > s}
    misaligned = []
    ents_sorted = sorted(ents, key=lambda x: (x[0], x[1]))
    k = 0
    for ent in ents_sorted:
        s, e, lab = ent[0], ent[1], ent[2]
        li = lab_idx[lab]
        if s not in starts or e not in ends:
            misaligned.append(ent)
        while k < len(spans) and spans[k][1] <= s:
            k += 1
        j, first = k, True
        while j < len(spans) and spans[j][0] < e:
            if spans[j][1] > spans[j][0]:                            # skip empty (continuation-byte) tokens
                if tag[j] == 0:                                     # never overwrite another entity's token
                    tag[j] = 1 + 2 * li if first else 2 + 2 * li
                first = False
            j += 1
    return tag, misaligned


def token_tags_to_spans(spans, tag, labels):
    """BIO tag ids -> [(start_char, end_char, label)] with lenient repair: an I- tag that does not continue an
    entity of the same label starts a new one."""
    out, cur = [], None
    for (s, e), t in zip(spans, tag):
        if e <= s or t == IGNORE:
            continue
        if t == 0:
            if cur:
                out.append(tuple(cur)); cur = None
            continue
        li, is_b = (t - 1) // 2, (t - 1) % 2 == 0
        lab = labels[li]
        if cur and not is_b and cur[2] == lab:
            cur[1] = e
        else:
            if cur:
                out.append(tuple(cur))
            cur = [s, e, lab]
    if cur:
        out.append(tuple(cur))
    return out


# ---------------- boundary repair (character-level, after token decoding) ----------------
TRAIL_PUNCT = set(",.;:)]}'\"’”`-/")
LEAD_PUNCT = set("([{'\"‘“`-:./")


def learn_trim_table(docs, golds, tok_spans_by_doc, labels):
    """Data-driven boundary repair learned on TRAINING docs only. For each (label, trailing char) seen at the end of
    an oracle token span, count how often the gold entity excludes that char -> trim if excluded > kept.
    Same for leading chars. Returns (trail_rule, lead_rule) dicts {(label, ch): True/False}."""
    from collections import Counter
    t_ex, t_keep, l_ex, l_keep = Counter(), Counter(), Counter(), Counter()
    for did, ents in golds.items():
        text, spans = docs[did], tok_spans_by_doc[did]
        tag, _ = gold_to_token_tags(spans, ents, labels)
        pred = token_tags_to_spans(spans, tag, labels)
        # match each oracle span to the gold entity it covers (same label, contained)
        gl = sorted((e[0], e[1], e[2]) for e in ents)
        for (s, e, lab) in pred:
            cands = [g for g in gl if g[2] == lab and g[0] >= s and g[1] <= e]
            if len(cands) != 1:
                continue
            gs, ge, _ = cands[0]
            # trailing chars of the token span beyond gold end
            for p in range(e - 1, s - 1, -1):
                ch = text[p]
                if ch not in TRAIL_PUNCT:
                    break
                if p >= ge:
                    t_ex[(lab, ch)] += 1
                else:
                    t_keep[(lab, ch)] += 1
                    break
            for p in range(s, e):
                ch = text[p]
                if ch not in LEAD_PUNCT:
                    break
                if p < gs:
                    l_ex[(lab, ch)] += 1
                else:
                    l_keep[(lab, ch)] += 1
                    break
    trail = {k: t_ex[k] > t_keep[k] for k in set(t_ex) | set(t_keep)}
    lead = {k: l_ex[k] > l_keep[k] for k in set(l_ex) | set(l_keep)}
    return trail, lead


def repair_span(text, s, e, lab, trail_rule, lead_rule):
    while e > s + 1 and text[e - 1] in TRAIL_PUNCT and trail_rule.get((lab, text[e - 1]), False):
        e -= 1
    while s < e - 1 and text[s] in LEAD_PUNCT and lead_rule.get((lab, text[s]), False):
        s += 1
    return s, e


def finalize_doc_entities(text, raw_spans, trail_rule=None, lead_rule=None):
    """Repair, drop empty/invalid, de-duplicate (the evaluator FAILS on duplicate (start,end,label)), sort."""
    seen, out = set(), []
    for s, e, lab in raw_spans:
        if trail_rule is not None:
            s, e = repair_span(text, s, e, lab, trail_rule, lead_rule or {})
        s, e = int(s), int(e)
        if not (0 <= s < e <= len(text)):
            continue
        key = (s, e, lab)
        if key in seen:
            continue
        seen.add(key)
        out.append([s, e, lab])
    out.sort()
    return out


# ---------------- strict scorer (same arithmetic as evaluate_phase2.py) + error breakdown ----------------
def strict_scores(golds, preds, labels):
    stats = {l: [0, 0, 0] for l in labels}
    for did, g in golds.items():
        gs = {(x[0], x[1], x[2]) for x in g}
        ps = {(x[0], x[1], x[2]) for x in preds.get(did, [])}
        for l in labels:
            a = {x for x in gs if x[2] == l}; b = {x for x in ps if x[2] == l}
            stats[l][0] += len(a & b); stats[l][1] += len(b - a); stats[l][2] += len(a - b)
    per = {}
    for l, (tp, fp, fn) in stats.items():
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        per[l] = dict(P=p, R=r, F1=f, tp=tp, fp=fp, fn=fn)
    n = len(labels)
    return dict(macro_F1=sum(v["F1"] for v in per.values()) / n,
                macro_P=sum(v["P"] for v in per.values()) / n,
                macro_R=sum(v["R"] for v in per.values()) / n, per_label=per)


def error_breakdown(golds, preds):
    """Classify each wrong prediction: boundary error (overlaps a gold entity of the same label), label error (same
    exact span, other label), boundary+label, or spurious (overlaps no gold entity). Also gold entities missed."""
    from collections import Counter
    c = Counter()
    for did, g in golds.items():
        gs = {(x[0], x[1], x[2]) for x in g}
        for (s, e, l) in {(x[0], x[1], x[2]) for x in preds.get(did, [])}:
            if (s, e, l) in gs:
                c["correct"] += 1; continue
            same_span = [x for x in gs if x[0] == s and x[1] == e]
            over = [x for x in gs if x[0] < e and s < x[1]]
            if same_span:
                c["label_error"] += 1
            elif any(x[2] == l for x in over):
                c["boundary_error"] += 1
            elif over:
                c["boundary_and_label_error"] += 1
            else:
                c["spurious"] += 1
        ps = {(x[0], x[1]) for x in preds.get(did, [])}
        for x in gs:
            if not any(p[0] < x[1] and x[0] < p[1] for p in ps):
                c["missed_entirely"] += 1
    return dict(c)


# ---------------- sliding windows ----------------
def make_windows(n_tokens, window, stride):
    """Start indices of windows of length `window` every `stride` tokens, the last one flush with the end."""
    if n_tokens <= window:
        return [0]
    starts = list(range(0, n_tokens - window + 1, stride))
    if starts[-1] + window < n_tokens:
        starts.append(n_tokens - window)
    return starts


## 6. Phase-1 model code (copied unchanged from moe-muonfinallyer-mbr-highsuperemecourt (Phase-1 notebook, cells 'Model code'))

In [6]:
NORM_TYPE = "layernorm"; NORM_POSITION = "pre"   # as in the Phase-1 run
CONTEXT_LEN = 2048

class RMSNorm(nn.Module):
    """Like LayerNorm, but simpler: it only rescales by the root-mean-square
    of the values, it does not re-center the mean. Cheaper, and used in
    models like LLaMA."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


def make_norm(dim, norm_type):
    """norm_type is "layernorm", "rmsnorm" or "deepnorm".
    deepnorm reuses a plain LayerNorm -- what makes deepnorm special isn't
    the norm formula, it's the residual scaling (alpha) and weight init
    (beta) we apply around it below."""
    if norm_type == "rmsnorm":
        return RMSNorm(dim)
    else:  # "layernorm" or "deepnorm"
        return nn.LayerNorm(dim)


def deepnorm_alpha(n_layers):
    """Residual scaling factor from the DeepNet paper: alpha = (2N)^0.25."""
    return (2.0 * n_layers) ** 0.25


def deepnorm_init(output_projections, n_layers):
    """Scales the init of a sub-layer's OUTPUT projection(s) (e.g. attention
    out_proj, the feed-forward's second linear) by beta = (8N)^-0.25, so a
    deep post-norm stack stays stable to train. Only used when
    NORM_TYPE == "deepnorm"."""
    beta = (8.0 * n_layers) ** -0.25
    for module in output_projections:
        nn.init.xavier_normal_(module.weight, gain=beta)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

# ---- Settings specific to this notebook ------------------------------
# ---- Mixture-of-Experts FFN (DeepSeekMoE style) -------------------------------------------------------------
MOE_N_ROUTED = 16          # routed experts per MoE layer (fine-grained: each is only 1/4 the width of the dense FFN)
MOE_N_SHARED = 1           # shared experts: ALWAYS applied to every token (they hold the common knowledge)
MOE_TOP_K = 3              # routed experts used per token. Active FFN width = (1 shared + 3 routed) x 768 = 3072 = the dense FFN's width,
                           # so the compute per token matches the dense GQA run; the TOTAL FFN capacity is 17 x 768 = 4.25x larger.
MOE_D_EXPERT = 768         # hidden width of one expert
MOE_DENSE_FIRST_LAYERS = 1 # the first layer stays a dense FFN (as in DeepSeek: early layers route poorly)
MOE_BIAS_RATE = 0.001      # aux-loss-free load balancing (DeepSeek-V3): per step, a per-expert selection bias moves by this much
                           # towards the under-used experts. The bias only affects WHICH experts are chosen, never the output weights.

GQA_GROUP_SIZE = 4      # query heads that share one key/value head. 4 is exactly
                        # what ATTENTION_TYPE = "gqa" meant in the original notebooks.
CACHE_HEADROOM = 512    # slots reserved inside CONTEXT_LEN for GENERATED tokens.
                        # The prompt is cut to its last CONTEXT_LEN - CACHE_HEADROOM
                        # tokens; if generation fills the cache, the window is
                        # re-prefilled (see the markdown above for why). With
                        # CONTEXT_LEN = 2048 that leaves room for a 1536-token prompt
                        # (a 512-word legal prompt is roughly 700-1000 tokens) AND
                        # ~512 generated tokens, so in practice nothing is cut.
USE_NATIVE_GQA = False  # decided automatically by a micro-benchmark in the next
                        # section (True = pass enable_gqa=True to SDPA instead of
                        # expanding K/V with repeat_interleave)
POSITIONAL_ENCODING = "rope"   # "rope" | "learned"
                        # rope    = rotary embeddings: Q and K are rotated by a
                        #           position-dependent angle inside attention, so
                        #           attention scores depend on RELATIVE distance
                        #           (RoFormer, Su et al. 2021). No position table.
                        # learned = a trainable CONTEXT_LEN-row position table added to
                        #           the token embeddings (GPT-2 style, what the original
                        #           decoder-only notebook used).
ROPE_BASE = 10000.0     # RoPE frequency base: theta_i = ROPE_BASE ** (-2i / head_dim)
assert POSITIONAL_ENCODING in ("rope", "learned"), "POSITIONAL_ENCODING must be 'rope' or 'learned'"


def build_rope_cache(max_seq_len, head_dim, base=10000.0):
    """cos/sin tables for RoPE, each (max_seq_len, head_dim). Split-half
    convention: channel i is paired with channel i + head_dim/2, so the angle
    table is cat([freqs, freqs]) (NOT repeat_interleave, which pairs neighbours
    instead). Same construction as the RoPE written for Assignment 3."""
    assert head_dim % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (2 * torch.arange(0, head_dim // 2).float() / head_dim))
    positions = torch.arange(0, max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)              # (max_seq_len, head_dim // 2)
    angles = torch.cat([freqs, freqs], dim=-1)            # (max_seq_len, head_dim)
    return angles.cos(), angles.sin()


def rotate_half(x):
    """Split the last dim in half and return (-x2, x1): a 90-degree rotation step."""
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    """x_rotated = x * cos + rotate_half(x) * sin. x: (..., head_dim); cos/sin
    already broadcastable against x (the model gathers them per position). The
    arithmetic is done in fp32 (even under fp16 autocast) and cast back, so the
    rotation stays accurate and q/k keep the same dtype as v for SDPA."""
    xf = x.float()
    return (xf * cos + rotate_half(xf) * sin).to(x.dtype)


class KVCache:
    """Pre-allocated key/value storage for EVERY layer, written in place (the
    'static cache' idea from gpt-fast). Shapes are (batch, n_kv_heads, max_len,
    head_dim) -- n_kv_heads, not n_heads, which is where GQA's 4x saving
    lives. `length` is how many positions are currently filled (the same for
    every row in the batch: shorter prompts are left-padded)."""

    def __init__(self, n_layers, batch_size, n_kv_heads, max_len, head_dim, device, dtype):
        shape = (batch_size, n_kv_heads, max_len, head_dim)
        self.k = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.v = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.length = 0
        self.max_len = max_len

    def reset(self):
        self.length = 0

    def num_bytes(self):
        return sum(t.numel() * t.element_size() for t in self.k + self.v)


class CachedGQAttention(nn.Module):
    """Grouped-query self-attention (causal) with two entry points:
      forward()      -- the parallel form used for training and for PREFILL.
                        Optionally also stores this call's keys/values in a cache.
      decode_step()  -- ONE new token, reading/writing the cache.
    Parameter names (q_proj/k_proj/v_proj/out_proj) match the original
    MultiHeadAttention with ATTENTION_TYPE="gqa", so checkpoints are interchangeable."""

    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.group_size = n_heads // n_kv_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(n_heads * self.head_dim, d_model)

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        """x: (batch, seq, d_model). key_padding_mask (optional): (batch, seq)
        bool, True = real (non-pad) position; leave None when nothing is
        padded so the fused kernels can run mask-free. If k_cache/v_cache are
        given, this call's (un-expanded) keys/values are written to
        positions [0:seq] of them -- that is the PREFILL step. rope (optional):
        (cos, sin), each (batch or 1, 1, seq, head_dim), gathered from the
        model's RoPE tables -- None when POSITIONAL_ENCODING == "learned"."""
        batch_size, seq_len, _ = x.shape
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if rope is not None:
            # rotate BEFORE caching: the cache stores already-rotated keys (each
            # key is rotated by its own position exactly once, when it is written)
            q = apply_rope(q, *rope)
            k = apply_rope(k, *rope)

        if k_cache is not None:
            k_cache[:, :, :seq_len] = k.to(k_cache.dtype)
            v_cache[:, :, :seq_len] = v.to(v_cache.dtype)

        attn_mask = None
        if key_padding_mask is not None:
            positions = torch.arange(seq_len, device=x.device)
            causal = positions.unsqueeze(1) >= positions.unsqueeze(0)               # (seq, seq)
            allowed = causal.unsqueeze(0) & key_padding_mask.unsqueeze(1)           # (batch, seq, seq)
            # a query may always see itself -- guarantees no row is fully
            # masked (which would produce NaNs for pad-position queries)
            allowed = allowed | torch.eye(seq_len, dtype=torch.bool, device=x.device).unsqueeze(0)
            attn_mask = allowed.unsqueeze(1)                                        # (batch, 1, seq, seq)

        use_causal_flag = attn_mask is None
        if USE_NATIVE_GQA and self.group_size > 1:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag, enable_gqa=True)
        else:
            if self.group_size > 1:
                k = k.repeat_interleave(self.group_size, dim=1)
                v = v.repeat_interleave(self.group_size, dim=1)
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag)

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.n_heads * self.head_dim)
        return self.out_proj(out)

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model) -- ONE new position per row. Writes its K/V at
        index cache_len and attends over cache positions 0..cache_len
        (inclusive). key_mask (optional): (batch, cache_len + 1) bool, True =
        real key; None when the batch has no padding. rope (optional): (cos,
        sin), each (batch, head_dim) for this step's position."""
        batch_size = x_t.shape[0]

        # Fold the `group_size` query heads that share one K/V head into the
        # query-LENGTH axis: (batch, n_kv_heads, group_size, head_dim). The
        # cache is then used as-is (never expanded), and because the new token
        # is the last position no causal mask is needed.
        q = self.q_proj(x_t).view(batch_size, self.n_kv_heads, self.group_size, self.head_dim)
        k_new = self.k_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)
        v_new = self.v_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)

        if rope is not None:
            cos, sin = rope
            q = apply_rope(q, cos.view(batch_size, 1, 1, -1), sin.view(batch_size, 1, 1, -1))
            k_new = apply_rope(k_new, cos.view(batch_size, 1, -1), sin.view(batch_size, 1, -1))

        k_cache[:, :, cache_len] = k_new.to(k_cache.dtype)
        v_cache[:, :, cache_len] = v_new.to(v_cache.dtype)
        k = k_cache[:, :, :cache_len + 1]
        v = v_cache[:, :, :cache_len + 1]
        if q.dtype != k.dtype:
            q = q.to(k.dtype)

        attn_mask = None if key_mask is None else key_mask.view(batch_size, 1, 1, -1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)   # (batch, n_kv, group, head_dim)
        out = out.reshape(batch_size, self.n_heads * self.head_dim).to(x_t.dtype)
        return self.out_proj(out)


class ExpertMLP(nn.Module):
    """One expert: the same Linear -> GELU -> Linear as the dense FFN, just narrower."""

    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.up = nn.Linear(d_model, d_hidden)
        self.down = nn.Linear(d_hidden, d_model)

    def forward(self, x):
        return self.down(F.gelu(self.up(x)))


class MoEFFN(nn.Module):
    """DeepSeekMoE feed-forward layer, written from scratch.

        y = sum_s Shared_s(x)  +  sum_{e in TopK(x)} g_e(x) * Routed_e(x)

    * router: affinity s_e = sigmoid(w_e . x) for every routed expert e (DeepSeek-V3 uses sigmoid, not softmax)
    * selection: the top-k experts by (s_e + bias_e); the bias is a NON-learned buffer used only for choosing
    * gates: the selected s_e, renormalised to sum to 1 (so the routed part has a fixed scale)
    * load balancing WITHOUT an auxiliary loss: in training, after each forward pass, bias_e moves by +-MOE_BIAS_RATE
      towards the under-loaded experts (overloaded: down, underloaded: up). Nothing is dropped (no capacity limit).
    Every token is processed independently, so the KV-cache decode path needs no change."""

    def __init__(self, d_model, n_routed, n_shared, top_k, d_expert, bias_rate):
        super().__init__()
        assert 1 <= top_k <= n_routed
        self.n_routed, self.top_k, self.bias_rate = n_routed, top_k, bias_rate
        self.router = nn.Linear(d_model, n_routed, bias=False)
        nn.init.normal_(self.router.weight, std=0.02)
        self.shared = nn.ModuleList([ExpertMLP(d_model, d_expert) for _ in range(n_shared)])
        self.experts = nn.ModuleList([ExpertMLP(d_model, d_expert) for _ in range(n_routed)])
        self.register_buffer("select_bias", torch.zeros(n_routed))      # saved in the checkpoint (it is part of the trained router)
        self.last_load = None                                           # tokens routed to each expert in the latest training step

    def forward(self, x):
        shape = x.shape
        flat = x.reshape(-1, shape[-1])                                  # (N, d): the router does not care about batch/position
        affinity = torch.sigmoid(self.router(flat).float())              # (N, E), fp32 even under autocast
        _, chosen = (affinity + self.select_bias).topk(self.top_k, dim=-1)       # (N, k): selection uses the bias ...
        gate = affinity.gather(1, chosen)                                # ... but the gate values do not
        gate = gate / gate.sum(dim=-1, keepdim=True)
        out = torch.zeros_like(flat)
        for expert in self.shared:
            out = out + expert(flat)
        for e, expert in enumerate(self.experts):
            token_idx, slot = (chosen == e).nonzero(as_tuple=True)       # the tokens that picked expert e, and in which of their k slots
            if token_idx.numel() == 0:
                continue
            y = expert(flat[token_idx]) * gate[token_idx, slot].unsqueeze(-1).to(flat.dtype)
            out.index_add_(0, token_idx, y.to(out.dtype))
        if self.training:
            with torch.no_grad():
                load = torch.bincount(chosen.reshape(-1), minlength=self.n_routed).float()
                self.last_load = load
                self.select_bias += self.bias_rate * torch.sign(load.mean() - load)
        return out.reshape(shape)


def moe_load_stats(model):
    """[(layer index, busiest expert's load / the average (1.0 = perfectly balanced), number of experts with < 10% of an even share)]
    for every MoE layer, from the latest training step."""
    rows = []
    for i, blk in enumerate(model.blocks):
        if isinstance(blk.ff, MoEFFN) and blk.ff.last_load is not None:
            l = blk.ff.last_load.cpu()
            rows.append((i, round((l.max() / l.mean()).item(), 3), int((l < 0.1 * l.mean()).sum().item())))
    return rows


def moe_load_report(model):
    rows = moe_load_stats(model)
    if not rows:
        print("MoE load report: no training step has run in this session yet.")
        return
    print("MoE load balance (max expert load / mean, experts with <10% of an even share):")
    print("  " + "   ".join(f"layer {i}: {r:.2f}x / {d}" for i, r, d in rows))


def count_active_params(model):
    """Parameters used per token: everything except the routed experts that are NOT selected (top_k of n_routed are used)."""
    total = sum(p.numel() for p in model.parameters())
    inactive = 0
    for blk in model.blocks:
        if isinstance(blk.ff, MoEFFN):
            per_expert = sum(p.numel() for p in blk.ff.experts[0].parameters())
            inactive += per_expert * (blk.ff.n_routed - blk.ff.top_k)
    return total, total - inactive


class CachedDecoderBlock(nn.Module):
    """Same block as DecoderOnlyBlock (same norm / deepnorm wiring), with a
    decode_step() that mirrors forward() for one position."""

    def __init__(self, d_model, n_heads, d_ff, n_layers, layer_idx=0):
        super().__init__()
        self.norm_position = "post" if NORM_TYPE == "deepnorm" else NORM_POSITION

        n_kv_heads = max(1, n_heads // GQA_GROUP_SIZE)
        self.attn = CachedGQAttention(d_model, n_heads, n_kv_heads)
        if layer_idx >= MOE_DENSE_FIRST_LAYERS:
            assert NORM_TYPE != "deepnorm", "the MoE variant is only set up for layernorm/rmsnorm"
            self.ff = MoEFFN(d_model, MOE_N_ROUTED, MOE_N_SHARED, MOE_TOP_K, MOE_D_EXPERT, MOE_BIAS_RATE)
        else:
            self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1 = make_norm(d_model, NORM_TYPE)
        self.norm2 = make_norm(d_model, NORM_TYPE)

        if NORM_TYPE == "deepnorm":
            self.alpha = deepnorm_alpha(n_layers)
            deepnorm_init([self.attn.out_proj, self.ff[2]], n_layers)
        else:
            self.alpha = 1.0

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        if self.norm_position == "pre":
            x = x + self.attn(self.norm1(x), key_padding_mask=key_padding_mask,
                              k_cache=k_cache, v_cache=v_cache, rope=rope)
            x = x + self.ff(self.norm2(x))
        else:  # "post" (also deepnorm, with the alpha residual scale)
            x = self.norm1(x * self.alpha + self.attn(x, key_padding_mask=key_padding_mask,
                                                       k_cache=k_cache, v_cache=v_cache, rope=rope))
            x = self.norm2(x * self.alpha + self.ff(x))
        return x

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model). Norms and the FFN act per position, so only
        the attention part needs the cache."""
        if self.norm_position == "pre":
            x_t = x_t + self.attn.decode_step(self.norm1(x_t), k_cache, v_cache, cache_len, key_mask, rope)
            x_t = x_t + self.ff(self.norm2(x_t))
        else:
            x_t = self.norm1(x_t * self.alpha + self.attn.decode_step(x_t, k_cache, v_cache, cache_len, key_mask, rope))
            x_t = self.norm2(x_t * self.alpha + self.ff(x_t))
        return x_t


class CachedDecoderModel(nn.Module):
    """GPT-style decoder, GQA attention, KV-cache aware. Positions come from
    POSITIONAL_ENCODING: "learned" (pos_emb table added to the token
    embeddings) or "rope" (rotations inside attention, no table)."""

    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, context_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.use_rope = (POSITIONAL_ENCODING == "rope")
        if self.use_rope:
            cos, sin = build_rope_cache(context_len, d_model // n_heads, ROPE_BASE)
            # tables, not parameters; persistent=False keeps them out of state_dict
            # (one pair shared by every layer -- they depend only on position)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        else:
            self.pos_emb = nn.Embedding(context_len, d_model)
        self.blocks = nn.ModuleList([
            CachedDecoderBlock(d_model, n_heads, d_ff, n_layers, layer_idx=i) for i in range(n_layers)
        ])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def _embed(self, token_ids, position_ids):
        """Token embedding (+ learned position embedding when not using RoPE)."""
        h = self.token_emb(token_ids)
        if not self.use_rope:
            h = h + self.pos_emb(position_ids)
        return h

    def forward(self, x, key_padding_mask=None, position_ids=None, last_position_only=False):
        """Training / full-recompute forward. x: (batch, seq) token ids.
        position_ids (optional, (batch, seq)) lets padded batches give each
        row's first real token position 0. last_position_only=True skips the
        big vocab projection for every position but the last (used by the
        uncached reference generator)."""
        batch_size, seq_len = x.shape
        if position_ids is None:
            position_ids = torch.arange(seq_len, device=x.device).unsqueeze(0)
        h = self._embed(x, position_ids)
        rope = None
        if self.use_rope:
            cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]   # (batch or 1, seq, head_dim)
            rope = (cos.unsqueeze(1), sin.unsqueeze(1))                            # add the heads axis
        for block in self.blocks:
            h = block(h, key_padding_mask=key_padding_mask, rope=rope)
        if last_position_only:
            h = h[:, -1, :]
        return self.fc_out(h)

    def prefill(self, x, pad_lens, cache):
        """PREFILL. x: (batch, P) token ids, LEFT-padded; pad_lens: list of
        ints (pad count per row). Fills cache positions [0:P] for every layer,
        sets cache.length = P, and returns ONLY the last position's logits
        (batch, vocab) -- the one thing generation needs."""
        batch_size, prefill_len = x.shape
        assert prefill_len <= cache.max_len, "prompt window longer than the cache"
        pad_t = torch.tensor(pad_lens, dtype=torch.long, device=x.device)
        arange = torch.arange(prefill_len, device=x.device).unsqueeze(0)
        position_ids = (arange - pad_t.unsqueeze(1)).clamp(min=0)   # first REAL token -> position 0
        key_padding_mask = (arange >= pad_t.unsqueeze(1)) if any(p > 0 for p in pad_lens) else None

        h = self._embed(x, position_ids)
        rope = None
        if self.use_rope:
            cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]
            rope = (cos.unsqueeze(1), sin.unsqueeze(1))
        for layer_idx, block in enumerate(self.blocks):
            h = block(h, key_padding_mask=key_padding_mask,
                      k_cache=cache.k[layer_idx], v_cache=cache.v[layer_idx], rope=rope)
        cache.length = prefill_len
        return self.fc_out(h[:, -1, :])

    def decode_step(self, token_ids, pad_t, has_padding, cache):
        """DECODE. token_ids: (batch,) -- ONE new token per row. pad_t: (batch,)
        long tensor of pad counts. Appends to the cache, returns logits
        (batch, vocab) for the NEXT token. O(1) new work per step (plus one
        read of the cache)."""
        cache_len = cache.length
        assert cache_len < cache.max_len, "cache is full -- re-prefill before decoding further"
        position_ids = (cache_len - pad_t).clamp(min=0)              # (batch,)
        key_mask = None
        if has_padding:
            key_mask = torch.arange(cache_len + 1, device=token_ids.device).unsqueeze(0) >= pad_t.unsqueeze(1)

        h = self._embed(token_ids, position_ids)
        rope = (self.rope_cos[position_ids], self.rope_sin[position_ids]) if self.use_rope else None   # each (batch, head_dim)
        for layer_idx, block in enumerate(self.blocks):
            h = block.decode_step(h, cache.k[layer_idx], cache.v[layer_idx], cache_len, key_mask, rope)
        cache.length = cache_len + 1
        return self.fc_out(h)

## 7. NER backbone wrapper (hidden states instead of next-token logits; optional bidirectional / prefix attention)

In [7]:
# ===================== Backbone wrapper for the Phase-1 decoder models (GQA / MoE CachedDecoderModel) =====================
# The Phase-1 model code above is UNCHANGED. For NER we only (a) run its blocks and take the hidden states instead of the
# next-token logits, and (b) optionally replace the attention's forward with one that can drop the causal mask
# ("bidirectional") and/or prepend learned prefix keys/values (prefix tuning). With both options off it calls the
# original forward unchanged.

_ORIG_GQA_FORWARD = CachedGQAttention.forward


def ner_gqa_forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
    bidir = getattr(self, "ner_bidirectional", False)
    pk = getattr(self, "prefix_k", None)
    if not bidir and pk is None:
        return _ORIG_GQA_FORWARD(self, x, key_padding_mask=key_padding_mask, k_cache=k_cache, v_cache=v_cache, rope=rope)
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
    if rope is not None:
        q = apply_rope(q, *rope)
        k = apply_rope(k, *rope)
    P = 0
    if pk is not None:                                          # learned prefix keys/values (no position: never rotated)
        P = pk.shape[1]
        k = torch.cat([pk.unsqueeze(0).expand(B, -1, -1, -1).to(k.dtype), k], dim=2)
        v = torch.cat([self.prefix_v.unsqueeze(0).expand(B, -1, -1, -1).to(v.dtype), v], dim=2)
    if self.group_size > 1:
        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)
    if bidir and P == 0 and key_padding_mask is None:          # full-attention fast path (no mask tensor at all)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.head_dim)
        return self.out_proj(out)
    pos = torch.arange(T, device=x.device)
    allowed = torch.ones(T, T, dtype=torch.bool, device=x.device) if bidir else (pos.unsqueeze(1) >= pos.unsqueeze(0))
    allowed = allowed.unsqueeze(0).expand(B, T, T)
    if key_padding_mask is not None:
        allowed = allowed & key_padding_mask.unsqueeze(1)
        allowed = allowed | torch.eye(T, dtype=torch.bool, device=x.device).unsqueeze(0)
    if P:
        allowed = torch.cat([torch.ones(B, T, P, dtype=torch.bool, device=x.device), allowed], dim=2)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed.unsqueeze(1))
    out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.head_dim)
    return self.out_proj(out)


CachedGQAttention.forward = ner_gqa_forward


class DecoderBackbone(nn.Module):
    def __init__(self, lm, bidirectional=False):
        super().__init__()
        self.original_param_count = sum(p.numel() for p in lm.parameters())
        self.lm = lm
        self.causal = not bidirectional
        for blk in lm.blocks:
            blk.attn.ner_bidirectional = bidirectional
        if isinstance(lm.fc_out, nn.Sequential):                # the final LayerNorm is kept; the vocab projection is dropped
            self.final_norm = lm.fc_out[0]
        else:
            self.final_norm = nn.Identity()
        lm.fc_out = nn.Identity()
        self.d_model = lm.token_emb.embedding_dim
        self.max_len = lm.rope_cos.shape[0] if lm.use_rope else lm.pos_emb.num_embeddings

    def layer_list(self):
        return list(self.lm.blocks)

    def final_modules_params(self):
        return list(self.final_norm.parameters())

    def embedding_weight(self):
        return self.lm.token_emb.weight

    def adapter_sites(self, kind):
        d = self.d_model
        if kind == "houlsby":                                   # after attention AND after the feed-forward, every block
            return [(m, d) for blk in self.lm.blocks for m in (blk.attn, blk.ff)]
        if kind == "pfeiffer":                                  # after the feed-forward only
            return [(blk.ff, d) for blk in self.lm.blocks]
        raise ValueError(kind)

    def enable_prefix(self, P, hidden=0):
        for blk in self.lm.blocks:
            a = blk.attn
            a.prefix_k = nn.Parameter(torch.randn(a.n_kv_heads, P, a.head_dim) * 0.02)
            a.prefix_v = nn.Parameter(torch.randn(a.n_kv_heads, P, a.head_dim) * 0.02)

    def features(self, ids, mask, soft_prompt=None):
        lm = self.lm
        h = lm.token_emb(ids)
        B = ids.shape[0]
        P = 0
        if soft_prompt is not None:
            P = soft_prompt.shape[0]
            h = torch.cat([soft_prompt.unsqueeze(0).expand(B, -1, -1).to(h.dtype), h], dim=1)
            mask = torch.cat([torch.ones(B, P, dtype=torch.bool, device=mask.device), mask], dim=1)
        T = h.shape[1]
        assert T <= self.max_len, f"window ({T} incl. {P} prompt tokens) longer than the model's {self.max_len} positions"
        pos = torch.arange(T, device=ids.device)
        rope = None
        if lm.use_rope:
            rope = (lm.rope_cos[pos].unsqueeze(0).unsqueeze(0), lm.rope_sin[pos].unsqueeze(0).unsqueeze(0))
        else:
            h = h + lm.pos_emb(pos).unsqueeze(0)
        # right padding: with causal attention real tokens never see the pads, so no mask is needed (fast fused kernel);
        # bidirectional attention needs the padding mask
        kpm = None if (self.causal or bool(mask.all())) else mask
        for blk in lm.blocks:
            h = blk(h, key_padding_mask=kpm, rope=rope)
        h = self.final_norm(h)
        return h[:, P:]


@torch.no_grad()
def lm_sanity_loss(lm, docs, device, n_windows=6, length=1024):
    """Next-token loss of the loaded Phase-1 model on a few NER-document windows (BEFORE the LM head is dropped).
    A correctly loaded model gives roughly its Phase-1 held-out loss (~2.5-3.5 nats); random weights give ~10.6."""
    lm.eval().to(device)
    tot, cnt = 0.0, 0
    for d in list(docs.values())[:n_windows]:
        ids = torch.tensor(d["ids"][:length + 1], dtype=torch.long, device=device).unsqueeze(0)
        if ids.shape[1] < 2:
            continue
        logits = lm(ids[:, :-1]).float()
        tot += F.cross_entropy(logits.view(-1, logits.shape[-1]), ids[:, 1:].reshape(-1), reduction="sum").item()
        cnt += ids.shape[1] - 1
    return tot / max(cnt, 1)


def build_phase1_lm():
    """The Phase-1 MoE exactly as trained (218.7M params), with its trained weights."""
    torch.manual_seed(0)
    lm = CachedDecoderModel(VOCAB, d_model=768, n_heads=12, n_layers=8, d_ff=3072, context_len=CONTEXT_LEN)
    lm.fc_out = nn.Sequential(make_norm(768, NORM_TYPE), lm.fc_out)          # the final LayerNorm (FINAL_NORM = True in Phase 1)
    lm.load_state_dict(PHASE1_STATE, strict=True)
    for m in lm.modules():
        if isinstance(m, MoEFFN):
            m.bias_rate = MOE_BIAS_RATE_FT
    return lm


def make_backbone(lm):
    return DecoderBackbone(lm, bidirectional=(ATTENTION == "bidirectional"))


## 8. NER engine: head, adaptation methods (full FT, partial FT, BitFit, LoRA, LoRMA, adapters, prompt, prefix), training, inference

In [8]:
# ===================== Phase 2 L-NER engine: backbone wrapper, NER head, adaptation methods, training, inference =====================
# Works with any Phase-1 model wrapped as a "backbone" exposing  features(ids, mask, soft_prompt) -> (B, T, d_model).
import math, time, random, json, os, copy
import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------------------------------------------------
# 1. Documents -> token ids, character spans, BIO tags; windows
# ----------------------------------------------------------------------------------------------------------------------
def prepare_docs(tok, texts, golds, labels, mode):
    docs = {}
    for did, text in texts.items():
        ids, spans = encode_with_offsets(tok, text, mode)
        tags = gold_to_token_tags(spans, golds[did], labels)[0] if golds is not None else None
        docs[did] = dict(text=text, ids=ids, spans=spans, tags=tags)
    return docs


def window_items(docs, window, stride):
    items = []
    for did, d in docs.items():
        n = len(d["ids"])
        if n == 0:
            continue
        for s in make_windows(n, window, stride):
            items.append((did, s))
    return items


def collate(docs, items, window, pad_id=0, with_labels=True):
    L = max(min(window, len(docs[did]["ids"]) - s) for did, s in items)
    B = len(items)
    ids = torch.full((B, L), pad_id, dtype=torch.long)
    mask = torch.zeros((B, L), dtype=torch.bool)
    labels = torch.full((B, L), IGNORE, dtype=torch.long) if with_labels else None
    for b, (did, s) in enumerate(items):
        seg = docs[did]["ids"][s:s + window]
        n = len(seg)
        ids[b, :n] = torch.tensor(seg, dtype=torch.long)
        mask[b, :n] = True
        if with_labels:
            labels[b, :n] = torch.tensor(docs[did]["tags"][s:s + window], dtype=torch.long)
    return ids, mask, labels


# ----------------------------------------------------------------------------------------------------------------------
# 2. NER model = backbone + (optional soft prompt) + token-classification head
# ----------------------------------------------------------------------------------------------------------------------
class BiLSTMLayer(nn.Module):
    """A bidirectional LSTM over the backbone's hidden states (a prediction-head option). It gives every token the
    context to its RIGHT inside the window without changing the pretrained (causal) layers. Padding is packed away
    so the backward direction never reads pad positions."""

    def __init__(self, d_in, hidden):
        super().__init__()
        self.lstm = nn.LSTM(d_in, hidden, batch_first=True, bidirectional=True)
        self.out_dim = 2 * hidden

    def forward(self, h, mask):
        lengths = mask.sum(1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(h.float(), lengths, batch_first=True, enforce_sorted=False)
        with torch.autocast(device_type=h.device.type, enabled=False):
            out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=h.shape[1])
        return out


class NERModel(nn.Module):
    def __init__(self, backbone, d_model, n_tags, head_type="linear", head_dropout=0.1, head_norm=False, lstm_hidden=256):
        super().__init__()
        self.backbone = backbone
        self.soft_prompt = None                          # set by the prompt-tuning technique
        self.head_type = head_type
        self.bilstm = None
        layers = [nn.Dropout(head_dropout)]
        if head_norm:
            layers.append(nn.LayerNorm(d_model))
        if head_type == "linear":
            layers.append(nn.Linear(d_model, n_tags))
        elif head_type == "mlp":
            layers += [nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(head_dropout), nn.Linear(d_model, n_tags)]
        elif head_type == "bilstm":
            self.bilstm = BiLSTMLayer(d_model, lstm_hidden)
            layers += [nn.Linear(self.bilstm.out_dim, n_tags)]
        else:
            raise ValueError(head_type)
        self.head = nn.Sequential(*layers)

    def head_parameters(self):
        ps = list(self.head.parameters())
        if self.bilstm is not None:
            ps += list(self.bilstm.parameters())
        return ps

    def forward(self, ids, mask):
        h = self.backbone.features(ids, mask, self.soft_prompt)
        if self.bilstm is not None:
            h = self.bilstm(self.head[0](h), mask)
            return self.head[1:](h)
        return self.head(h)


# ----------------------------------------------------------------------------------------------------------------------
# 3. Adaptation methods
# ----------------------------------------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """LoRA (Hu et al. 2022): y = W0 x + b + (alpha/r) * B A x.  A: (r, in) random, B: (out, r) zeros -> starts == W0."""

    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base = base
        self.r, self.scale = r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)


_ROT_IDX = {}


def _row_rotation_index(d, device):
    key = (d, str(device))
    if key not in _ROT_IDX:
        i = torch.arange(d, device=device).unsqueeze(1)
        j = torch.arange(d, device=device).unsqueeze(0)
        _ROT_IDX[key] = (j - i) % d                      # Mpi[i, j] = M[i, (j - i) mod d]  (row i rotated right by i)
    return _ROT_IDX[key]


class LoRMALinear(nn.Module):
    """LoRMA (Bihany, Patel, Modi, ACL Findings 2025): a MULTIPLICATIVE low-rank update of a frozen W0 (out x in).
      variant "plus" (LoRMA+, additive rank inflation):  h = ((alpha/r) B A + I) W0 x + b
            A: (r, out) random, B: (out, r) zeros  -> starts exactly at W0.  Reordered as  y0 = W0 x;  y0 + (alpha/r) B (A y0)
            so the out x out matrix is never formed (same cost order as LoRA).
      variant "pi" (LoRMA_pi, permutation-based inflation):  h = I_pi(B A) W0 x + b, where I_pi rotates row i of BA
            cyclically by i. Init: B[:, 0] = 1, other B entries random; A[0, 0] = 1, other A entries 0 -> I_pi(BA) = I.
            Needs the out x out matrix every step (O(d^2 (r + batch)))."""

    def __init__(self, base, r, alpha, variant="plus"):
        super().__init__()
        self.base = base
        self.variant = variant
        self.r, self.scale = r, alpha / r
        d = base.out_features
        if variant == "plus":
            self.A = nn.Parameter(torch.empty(r, d))
            self.B = nn.Parameter(torch.zeros(d, r))
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        elif variant == "pi":
            A = torch.zeros(r, d); A[0, 0] = 1.0
            B = torch.randn(d, r) * (1.0 / math.sqrt(d)); B[:, 0] = 1.0
            self.A, self.B = nn.Parameter(A), nn.Parameter(B)
        else:
            raise ValueError(variant)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        if self.variant == "plus":
            y = y0 + self.scale * F.linear(F.linear(y0, self.A), self.B)
        else:
            M = self.B @ self.A                                                        # (d, d)
            Mpi = torch.gather(M, 1, _row_rotation_index(M.shape[0], M.device))
            y = F.linear(y0, Mpi.to(y0.dtype))
        if self.base.bias is not None:
            y = y + self.base.bias
        return y


class BottleneckAdapter(nn.Module):
    """Houlsby/Pfeiffer bottleneck: out + up(act(down(out))); up is zero-initialised -> starts as identity."""

    def __init__(self, d, m):
        super().__init__()
        self.down = nn.Linear(d, m)
        self.up = nn.Linear(m, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


def _replace_linears(root, name_suffixes, make):
    """Replace every nn.Linear whose qualified name ends with one of name_suffixes by make(linear). Returns names."""
    done = []
    for qual, mod in list(root.named_modules()):
        for child_name, child in list(mod.named_children()):
            full = f"{qual}.{child_name}" if qual else child_name
            if isinstance(child, nn.Linear) and any(full.endswith(s) for s in name_suffixes):
                setattr(mod, child_name, make(child))
                done.append(full)
    return done


def apply_technique(ner, C, log=print):
    """Freezes/unfreezes and inserts modules according to C["TECHNIQUE"]. The head is always trainable.
    Returns a short description dict (for the results file)."""
    tech = C["TECHNIQUE"]
    bb = ner.backbone
    info = dict(technique=tech)
    for p in bb.parameters():
        p.requires_grad = False
    if tech == "full_ft":
        for p in bb.parameters():
            p.requires_grad = True
    elif tech == "partial_ft":
        k = C["PARTIAL_TOP_BLOCKS"]
        blocks = bb.layer_list()
        for blk in blocks[len(blocks) - k:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in bb.final_modules_params():
            p.requires_grad = True
        info.update(top_blocks=k, of_blocks=len(blocks))
    elif tech == "bitfit":
        n = 0
        for name, p in bb.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad = True; n += 1
        info.update(bias_tensors=n)
    elif tech == "lora":
        done = _replace_linears(bb, C["PEFT_TARGETS"], lambda lin: LoRALinear(lin, C["RANK"], C["ALPHA"], C.get("LORA_DROPOUT", 0.0)))
        info.update(targets=len(done), rank=C["RANK"], alpha=C["ALPHA"])
    elif tech == "lorma":
        done = _replace_linears(bb, C["PEFT_TARGETS"], lambda lin: LoRMALinear(lin, C["RANK"], C["ALPHA"], C["LORMA_VARIANT"]))
        info.update(targets=len(done), rank=C["RANK"], alpha=C["ALPHA"], variant=C["LORMA_VARIANT"])
    elif tech == "adapter":
        sites = bb.adapter_sites(C["ADAPTER_TYPE"])            # list of (module whose output gets an adapter, d)
        ner.adapters = nn.ModuleList()
        for mod, d in sites:
            ad = BottleneckAdapter(d, C["ADAPTER_DIM"])
            ner.adapters.append(ad)
            mod.register_forward_hook(lambda m, inp, out, ad=ad: ad(out))
        info.update(adapter_type=C["ADAPTER_TYPE"], sites=len(sites), bottleneck=C["ADAPTER_DIM"])
    elif tech == "prompt":
        P = C["PROMPT_LEN"]
        emb = bb.embedding_weight()
        idx = torch.randint(100, emb.shape[0], (P,), generator=torch.Generator().manual_seed(0))
        ner.soft_prompt = nn.Parameter(emb[idx].detach().clone().float())
        info.update(prompt_len=P)
    elif tech == "prefix":
        P = C["PREFIX_LEN"]
        bb.enable_prefix(P, C.get("PREFIX_HIDDEN", 0))
        info.update(prefix_len=P)
    else:
        raise ValueError(f"unknown TECHNIQUE {tech}")
    for p in ner.head_parameters():
        p.requires_grad = True
    total = sum(p.numel() for p in ner.parameters())
    trainable = sum(p.numel() for p in ner.parameters() if p.requires_grad)
    backbone_orig = bb.original_param_count
    info.update(params_total=total, params_trainable=trainable, params_backbone_original=backbone_orig,
                trainable_pct_of_backbone=round(100.0 * trainable / backbone_orig, 4))
    log(f"Technique {tech}: trainable {trainable:,} of {total:,} parameters ({info['trainable_pct_of_backbone']}% of the Phase-1 model's {backbone_orig:,})")
    return info


def trainable_state(ner):
    """Everything that can differ from the Phase-1 weights: trainable params + routing-bias buffers (if updated)."""
    st = {n: p.detach().to("cpu", copy=True) for n, p in ner.named_parameters() if p.requires_grad}
    for n, b in ner.named_buffers():
        if n.endswith("select_bias"):
            st[n] = b.detach().to("cpu", copy=True)
    return st


def load_trainable_state(ner, st):
    missing = [k for k in st if k not in dict(ner.named_parameters()) and k not in dict(ner.named_buffers())]
    assert not missing, f"state has unknown keys: {missing[:5]}"
    ner.load_state_dict(st, strict=False)


# ----------------------------------------------------------------------------------------------------------------------
# 4. Training (optionally with a knowledge-distillation teacher)
# ----------------------------------------------------------------------------------------------------------------------
def tag_class_weights(docs, n_tags, scheme):
    if scheme == "none":
        return None
    cnt = torch.zeros(n_tags)
    for d in docs.values():
        for t in d["tags"]:
            if t >= 0:
                cnt[t] += 1
    cnt = cnt.clamp(min=1)
    if scheme == "inv_sqrt":
        w = (cnt.sum() / cnt).sqrt()
    elif scheme == "inv":
        w = cnt.sum() / cnt
    else:
        raise ValueError(scheme)
    return w / w[0]                                         # O keeps weight 1


def _lr_lambda(total, warmup):
    def f(step):
        if step < warmup:
            return (step + 1) / max(1, warmup)
        return max(0.0, (total - step) / max(1, total - warmup))
    return f


def train_ner(ner, train_docs, C, device, eval_fn=None, teacher=None, log=print):
    """Trains the trainable parameters. eval_fn(ner) -> (score, details) is called after each epoch (if given); the
    best epoch's trainable state is kept. teacher(ids, mask) -> logits enables KD.
    Returns dict(history, best_epoch, best_score, best_state, train_seconds, peak_mem_gb, tokens_per_sec)."""
    ner.to(device)
    torch.manual_seed(C["SEED"]); random.seed(C["SEED"])
    items = window_items(train_docs, C["WINDOW"], C["TRAIN_STRIDE"])
    n_tags = ner.head[-1].out_features
    cw = tag_class_weights(train_docs, n_tags, C["CLASS_WEIGHTING"])
    cw = cw.to(device) if cw is not None else None
    head_params = [p for p in ner.head_parameters() if p.requires_grad]
    hp = set(id(p) for p in head_params)
    other = [p for p in ner.parameters() if p.requires_grad and id(p) not in hp]
    groups = [dict(params=head_params, lr=C["HEAD_LR"])]
    if other:
        groups.append(dict(params=other, lr=C["LR"]))
    opt = torch.optim.AdamW(groups, weight_decay=C["WEIGHT_DECAY"], betas=(0.9, 0.98))
    steps_per_epoch = math.ceil(len(items) / (C["BATCH"] * C["ACCUM"]))
    total = steps_per_epoch * C["EPOCHS"]
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda(total, int(C["WARMUP_FRAC"] * total)))
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    hist, best, best_state, best_ep, bad_epochs = [], -1.0, None, 0, 0
    t0, tokens = time.time(), 0
    log(f"training: {len(items)} windows/epoch, {steps_per_epoch} optimizer steps/epoch, {C['EPOCHS']} epochs max")
    for ep in range(1, C["EPOCHS"] + 1):
        ner.train()
        rng = random.Random(C["SEED"] * 1000 + ep)
        order = items[:]; rng.shuffle(order)
        run_loss, nb = 0.0, 0
        opt.zero_grad(set_to_none=True)
        for bi in range(0, len(order), C["BATCH"]):
            ids, mask, labels = collate(train_docs, order[bi:bi + C["BATCH"]], C["WINDOW"])
            ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = ner(ids, mask)
            logits = logits.float()
            loss = F.cross_entropy(logits.view(-1, n_tags), labels.view(-1), weight=cw, ignore_index=IGNORE)
            if teacher is not None and C.get("KD_ALPHA", 0) > 0:
                with torch.no_grad():
                    t_logits = teacher(ids, mask).float()
                keep = labels.view(-1) != IGNORE
                T = C["KD_TEMPERATURE"]
                kd = F.kl_div(F.log_softmax(logits.view(-1, n_tags)[keep] / T, -1),
                              F.softmax(t_logits.view(-1, n_tags)[keep] / T, -1), reduction="batchmean") * T * T
                loss = (1 - C["KD_ALPHA"]) * loss + C["KD_ALPHA"] * kd
            scaler.scale(loss / C["ACCUM"]).backward()
            nb += 1
            tokens += int(mask.sum())
            run_loss += loss.item()
            if nb % C["ACCUM"] == 0 or bi + C["BATCH"] >= len(order):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], C["GRAD_CLIP"])
                scaler.step(opt); scaler.update(); sched.step()
                opt.zero_grad(set_to_none=True)
        row = dict(epoch=ep, train_loss=round(run_loss / max(1, nb), 5), seconds=round(time.time() - t0, 1))
        if eval_fn is not None:
            score, det = eval_fn(ner)
            row.update(heldout_macroF1=round(score, 5), **det)
            if score > best:
                best, best_ep, best_state, bad_epochs = score, ep, trainable_state(ner), 0
            else:
                bad_epochs += 1
        hist.append(row)
        log(f"  epoch {ep}: " + ", ".join(f"{k}={v}" for k, v in row.items() if k != "epoch"))
        if eval_fn is not None and C["PATIENCE"] and bad_epochs >= C["PATIENCE"]:
            log(f"  early stop: no improvement for {bad_epochs} epochs")
            break
    if eval_fn is None:
        best_state, best_ep = trainable_state(ner), len(hist)
    secs = time.time() - t0
    peak = torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else None
    return dict(history=hist, best_epoch=best_ep, best_score=best, best_state=best_state, train_seconds=round(secs, 1),
                peak_mem_gb=round(peak, 3) if peak else None, tokens_per_sec=round(tokens / max(secs, 1e-9), 1))


# ----------------------------------------------------------------------------------------------------------------------
# 5. Inference: windows -> per-token logits (merged) -> tags -> character spans
# ----------------------------------------------------------------------------------------------------------------------
@torch.no_grad()
def doc_logits(ner, doc, C, device, causal):
    n = len(doc["ids"])
    n_tags = ner.head[-1].out_features
    if n == 0:
        return torch.zeros(0, n_tags)
    W, S = C["WINDOW"], C["INFER_STRIDE"]
    starts = make_windows(n, W, S)
    out = torch.zeros(n, n_tags)
    best = torch.full((n,), -1.0)
    cnt = torch.zeros(n)
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    ner.eval()
    for bi in range(0, len(starts), C["INFER_BATCH"]):
        chunk = [("_", s) for s in starts[bi:bi + C["INFER_BATCH"]]]
        ids, mask, _ = collate({"_": doc}, chunk, W, with_labels=False)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            lg = ner(ids.to(device), mask.to(device)).float().cpu()
        for b, (_, s) in enumerate(chunk):
            L = int(mask[b].sum())
            pos = torch.arange(L)
            if C["MERGE"] == "average":
                out[s:s + L] += lg[b, :L]; cnt[s:s + L] += 1
            else:                                                # "context": keep the window giving each token most context
                score = pos.float() if causal else torch.minimum(pos, L - 1 - pos).float()
                upd = score > best[s:s + L]
                out[s:s + L][upd] = lg[b, :L][upd]
                best[s:s + L] = torch.where(upd, score, best[s:s + L])
    if C["MERGE"] == "average":
        out = out / cnt.clamp(min=1).unsqueeze(1)
    return out


def bio_allowed(n_tags):
    """allowed[prev, cur]: I-X may only follow B-X or I-X."""
    a = torch.ones(n_tags, n_tags, dtype=torch.bool)
    for cur in range(1, n_tags):
        if (cur - 1) % 2 == 1:                                    # I- tag
            b_same, i_same = cur - 1, cur
            for prev in range(n_tags):
                if prev not in (b_same, i_same):
                    a[prev, cur] = False
    return a


def viterbi_bio(logits):
    """Best tag sequence under BIO constraints (no transition scores, only the hard constraint)."""
    n, K = logits.shape
    if n == 0:
        return []
    lp = F.log_softmax(logits.float(), -1)
    allowed = bio_allowed(K)
    trans = torch.where(allowed, torch.tensor(0.0), torch.tensor(-1e9))
    start_ok = torch.tensor([0.0] + [0.0 if (k - 1) % 2 == 0 else -1e9 for k in range(1, K)])
    score = lp[0] + start_ok
    back = torch.zeros(n, K, dtype=torch.long)
    for t in range(1, n):
        cand = score.unsqueeze(1) + trans                         # (prev, cur)
        best, arg = cand.max(0)
        back[t] = arg
        score = best + lp[t]
    path = [int(score.argmax())]
    for t in range(n - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return path[::-1]


def decode_doc(doc, logits, labels, C, trail=None, lead=None):
    if len(doc["ids"]) == 0:
        return []
    if C["DECODE"] == "viterbi":
        tags = viterbi_bio(logits)
    else:
        tags = logits.argmax(-1).tolist()
    spans = doc["spans"]
    tags = [IGNORE if e <= s else t for (s, e), t in zip(spans, tags)]
    raw = token_tags_to_spans(spans, tags, labels)
    if C["BOUNDARY_REPAIR"]:
        return finalize_doc_entities(doc["text"], raw, trail, lead)
    return finalize_doc_entities(doc["text"], raw)


def predict_docs(ner, docs, labels, C, device, causal, trail=None, lead=None):
    t0 = time.time()
    preds = {did: decode_doc(d, doc_logits(ner, d, C, device, causal), labels, C, trail, lead) for did, d in docs.items()}
    return preds, time.time() - t0


def confusion_pairs(golds, preds):
    """(gold label -> predicted label) counts for predictions that overlap a gold entity of a different label."""
    from collections import Counter
    c = Counter()
    for did, g in golds.items():
        for p in preds.get(did, []):
            for x in g:
                if x[0] < p[1] and p[0] < x[1] and x[2] != p[2]:
                    c[f"{x[2]}->{p[2]}"] += 1
    return dict(c.most_common(25))


EVALUATOR_SRC = '#!/usr/bin/env python3\n"""Local evaluator for Phase 2.\n\nUsage:\n  python evaluate_phase2.py --truth truth.json --prediction prediction.json --labels labels.txt\n"""\nimport argparse, json, sys\n\nclass DuplicateJSONKeyError(ValueError): pass\ndef reject_constant(v): raise ValueError(f"Non-standard JSON numeric value \'{v}\' is not allowed.")\ndef pairs_no_duplicates(pairs):\n    out={}; dup=[]\n    for k,v in pairs:\n        if k in out: dup.append(k)\n        out[k]=v\n    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")\n    return out\ndef load_json(path):\n    with open(path,encoding=\'utf-8\') as f:\n        try: return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)\n        except json.JSONDecodeError as e: raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e\ndef labels(path):\n    with open(path,encoding=\'utf-8\') as f: xs=[x.strip() for x in f if x.strip()]\n    if not xs: raise ValueError(\'labels.txt is empty.\')\n    if len(xs)!=12: raise ValueError(f\'Expected 12 labels, found {len(xs)}.\')\n    if len(set(xs))!=len(xs): raise ValueError(\'labels.txt contains duplicate labels.\')\n    return xs\ndef spans(x, allowed, where):\n    if x is None: return set()\n    if not isinstance(x,list): raise ValueError(f\'{where}: expected a list of entities.\')\n    out=set()\n    for j,s in enumerate(x):\n        if not isinstance(s,list) or len(s) not in (3,4): raise ValueError(f\'{where}, entity {j}: expected [start,end,label] or [start,end,label,entity_text].\')\n        a,b,l=s[:3]\n        if isinstance(a,bool) or not isinstance(a,int) or isinstance(b,bool) or not isinstance(b,int): raise ValueError(f\'{where}, entity {j}: start/end must be integers.\')\n        if a<0 or b<=a: raise ValueError(f\'{where}, entity {j}: require 0 <= start < end.\')\n        if not isinstance(l,str) or l not in allowed: raise ValueError(f\'{where}, entity {j}: invalid label {l!r}.\')\n        if len(s)==4 and not isinstance(s[3],str): raise ValueError(f\'{where}, entity {j}: entity_text must be a string.\')\n        t=(a,b,l)\n        if t in out: raise ValueError(f\'{where}, entity {j}: duplicate span {t}.\')\n        out.add(t)\n    return out\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'Evaluate Phase 2 entity predictions on your own held-out split.\')\n    ap.add_argument(\'--truth\',required=True); ap.add_argument(\'--prediction\',required=True); ap.add_argument(\'--labels\',required=True)\n    args=ap.parse_args(); truth=load_json(args.truth); pred=load_json(args.prediction); labs=labels(args.labels)\n    if not isinstance(truth,dict) or not truth or not isinstance(pred,dict) or not pred: raise ValueError(\'Truth and prediction must be non-empty JSON objects.\')\n    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")\n    stats={l:[0,0,0] for l in labs}; allowed=set(labs)\n    for did in truth:\n        g=spans(truth[did],allowed,f\'truth[{did}]\'); p=spans(pred[did],allowed,f\'prediction[{did}]\')\n        for l in labs:\n            gs={x for x in g if x[2]==l}; ps={x for x in p if x[2]==l}\n            stats[l][0]+=len(gs&ps); stats[l][1]+=len(ps-gs); stats[l][2]+=len(gs-ps)\n    precisions=[]; recalls=[]; f1s=[]\n    print(f\'Examples: {len(truth)}\')\n    for l in labs:\n        tp,fp,fn=stats[l]; pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0; f=2*pr*rc/(pr+rc) if pr+rc else 0\n        precisions.append(pr); recalls.append(rc); f1s.append(f)\n        print(f\'{l}: P={pr:.6f} R={rc:.6f} F1={f:.6f}\')\n    print(f\'Strict Macro-F1:        {sum(f1s)/len(f1s):.6f}\')\n    print(f\'Strict Macro-Precision: {sum(precisions)/len(precisions):.6f}\')\n    print(f\'Strict Macro-Recall:    {sum(recalls)/len(recalls):.6f}\')\nif __name__==\'__main__\':\n    try: main()\n    except Exception as e: print(f\'EVALUATION FAILED: {e}\',file=sys.stderr); sys.exit(1)\n'

## 9. Tokenize the documents (with character offsets) and learn the boundary-repair tables

In [9]:
t0 = time.time()
TAG_NAMES = bio_tag_names(LABELS)
DOCS = {f: prepare_docs(bpe_tokenizer, TEXTS[f], GOLDS[f], LABELS, INPUT_MODE) for f in TEXTS}
TEST_DOCS = prepare_docs(bpe_tokenizer, TEST_TEXTS, None, LABELS, INPUT_MODE)
if INPUT_MODE == "none":
    check_encode_identical(bpe_tokenizer, {**TEXTS["fold1"], **TEXTS["fold2"], **TEST_TEXTS})
    print("token ids identical to BPETokenizer.encode on every document")
GOLD3 = {f: {d: [e[:3] for e in GOLDS[f][d]] for d in GOLDS[f]} for f in GOLDS}


def trim_tables(folds):
    tx, gd, sp = {}, {}, {}
    for f in folds:
        tx.update(TEXTS[f]); gd.update(GOLDS[f]); sp.update({d: DOCS[f][d]["spans"] for d in DOCS[f]})
    return learn_trim_table(tx, gd, sp, LABELS)


TRIM_CV = trim_tables([CV_TRAIN])                 # learned on the training fold only
TRIM_ALL = trim_tables(["fold1", "fold2"])        # for the test set
mis = sum(len(gold_to_token_tags(DOCS[f][d]["spans"], GOLDS[f][d], LABELS)[1]) for f in DOCS for d in DOCS[f])
print(f"{len(TAG_NAMES)} tags; {mis} gold entities not on token boundaries (INPUT_MODE={INPUT_MODE!r}); "
      f"windows/epoch on {CV_TRAIN}: {len(window_items(DOCS[CV_TRAIN], WINDOW, TRAIN_STRIDE))}; {time.time()-t0:.0f}s")


token ids identical to BPETokenizer.encode on every document
25 tags; 2828 gold entities not on token boundaries (INPUT_MODE='none'); windows/epoch on fold1: 182; 19s


## 10. Load the Phase-1 checkpoint and check it

In [10]:
t0 = time.time()
_ck = torch.load(fetch_model_file(PHASE1_CHECKPOINT), map_location="cpu", mmap=True)
PHASE1_STATE = _ck["model"] if "model" in _ck else _ck
del _ck
print(f"Phase-1 checkpoint loaded ({time.time()-t0:.0f}s): {PHASE1_CHECKPOINT}")

_lm = build_phase1_lm()
PHASE1_PARAMS = sum(p.numel() for p in _lm.parameters())
SANITY_LOSS = lm_sanity_loss(_lm, DOCS["fold1"], device)
print(f"Phase-1 model: {PHASE1_PARAMS:,} parameters; next-token loss on NER documents = {SANITY_LOSS:.3f} nats "
      f"(about 2.5-3.5 means the weights loaded correctly; ~{math.log(VOCAB):.1f} would mean random weights)")
assert SANITY_LOSS < 6.0, "the Phase-1 weights do not seem to be loaded correctly"
del _lm; gc.collect(); torch.cuda.empty_cache() if device.type == "cuda" else None


def build_ner():
    lm = build_phase1_lm()
    bb = make_backbone(lm)
    ner = NERModel(bb, bb.d_model, len(TAG_NAMES), HEAD_TYPE, HEAD_DROPOUT, HEAD_NORM, HEAD_LSTM_HIDDEN)
    info = apply_technique(ner, C)
    return ner.to(device), info


Phase-1 checkpoint loaded (5s): phase1_decoder_only_gqa_kvcache_moe_muon_finalnorm_continued2_highcourt_fp16_full_checkpoint.pt
Phase-1 model: 218,687,296 parameters; next-token loss on NER documents = 3.106 nats (about 2.5-3.5 means the weights loaded correctly; ~10.6 would mean random weights)


## 11. Fine-tune on fold1, select the epoch on fold2

In [11]:
# ---- train on CV_TRAIN, choose the epoch on CV_EVAL ----
RESULTS = dict(run=RUN_NAME, model=MODEL_KEY, phase1_checkpoint=PHASE1_CHECKPOINT, phase1_params=PHASE1_PARAMS,
               sanity_lm_loss=round(SANITY_LOSS, 4), settings=C, gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu")
ner, RESULTS["technique_info"] = build_ner()
causal = ATTENTION != "bidirectional" and HEAD_TYPE != "bilstm"   # merge rule: a BiLSTM head also uses right context


def eval_fn(model):
    preds, _ = predict_docs(model, DOCS[CV_EVAL], LABELS, C, device, causal, *TRIM_CV)
    s = strict_scores(GOLD3[CV_EVAL], preds, LABELS)
    return s["macro_F1"], dict(heldout_P=round(s["macro_P"], 4), heldout_R=round(s["macro_R"], 4))


cv = train_ner(ner, DOCS[CV_TRAIN], C, device, eval_fn=eval_fn)
load_trainable_state(ner, cv["best_state"])
RESULTS["cv"] = {k: v for k, v in cv.items() if k != "best_state"}
torch.save(cv["best_state"], f"{WORK}/{RUN_NAME}_cvmodel_state.pt")
print(f"best epoch {cv['best_epoch']}: held-out strict macro-F1 {cv['best_score']:.4f} | train {cv['train_seconds']}s, "
      f"peak GPU memory {cv['peak_mem_gb']} GB, {cv['tokens_per_sec']} tokens/s")


Technique full_ft: trainable 190,041,369 of 190,041,369 parameters (86.901% of the Phase-1 model's 218,687,296)
training: 182 windows/epoch, 23 optimizer steps/epoch, 10 epochs max
  epoch 1: train_loss=2.01852, seconds=39.3, heldout_macroF1=0.14753, heldout_P=0.1394, heldout_R=0.1733
  epoch 2: train_loss=0.50332, seconds=96.0, heldout_macroF1=0.26489, heldout_P=0.2141, heldout_R=0.3713
  epoch 3: train_loss=0.19807, seconds=152.6, heldout_macroF1=0.3666, heldout_P=0.3315, heldout_R=0.45
  epoch 4: train_loss=0.09661, seconds=208.8, heldout_macroF1=0.43138, heldout_P=0.3941, heldout_R=0.4898
  epoch 5: train_loss=0.05396, seconds=265.1, heldout_macroF1=0.4428, heldout_P=0.4312, heldout_R=0.4778
  epoch 6: train_loss=0.03586, seconds=321.3, heldout_macroF1=0.45602, heldout_P=0.4656, heldout_R=0.4677
  epoch 7: train_loss=0.02439, seconds=377.3, heldout_macroF1=0.46099, heldout_P=0.4747, heldout_R=0.4692
  epoch 8: train_loss=0.01703, seconds=433.4, heldout_macroF1=0.46163, heldout_P=0.

## 12. Held-out report (for the write-up)

In [12]:
# ---- detailed held-out evaluation of the selected model ----
preds_cv, infer_s = predict_docs(ner, DOCS[CV_EVAL], LABELS, C, device, causal, *TRIM_CV)
preds_norepair, _ = predict_docs(ner, DOCS[CV_EVAL], LABELS, dict(C, BOUNDARY_REPAIR=False), device, causal)
S = strict_scores(GOLD3[CV_EVAL], preds_cv, LABELS)
S0 = strict_scores(GOLD3[CV_EVAL], preds_norepair, LABELS)
n_tok = sum(len(d["ids"]) for d in DOCS[CV_EVAL].values())
RESULTS["heldout"] = dict(
    fold=CV_EVAL, macro_F1=round(S["macro_F1"], 4), macro_P=round(S["macro_P"], 4), macro_R=round(S["macro_R"], 4),
    macro_F1_without_boundary_repair=round(S0["macro_F1"], 4),
    per_label={l: {k: (round(v, 4) if isinstance(v, float) else v) for k, v in d.items()} for l, d in S["per_label"].items()},
    errors=error_breakdown(GOLD3[CV_EVAL], preds_cv), label_confusions=confusion_pairs(GOLD3[CV_EVAL], preds_cv),
    inference_seconds=round(infer_s, 1), inference_tokens_per_sec=round(n_tok / max(infer_s, 1e-9), 1))
print(f"held-out ({CV_EVAL}) strict macro-F1 {S['macro_F1']:.4f} (P {S['macro_P']:.4f}, R {S['macro_R']:.4f}); "
      f"without boundary repair {S0['macro_F1']:.4f}")
for l in LABELS:
    d = S["per_label"][l]
    print(f"  {l:10s} P={d['P']:.3f} R={d['R']:.3f} F1={d['F1']:.3f}  (tp {d['tp']}, fp {d['fp']}, fn {d['fn']})")
print("errors:", RESULTS["heldout"]["errors"])
print("most common label confusions:", dict(list(RESULTS["heldout"]["label_confusions"].items())[:8]))

# the official evaluator on the same predictions (must print the same macro-F1)
with open(f"{WORK}/evaluate_phase2.py", "w") as f:
    f.write(EVALUATOR_SRC)
json.dump(GOLDS[CV_EVAL], open(f"{WORK}/truth_{CV_EVAL}.json", "w"))
json.dump(preds_cv, open(f"{WORK}/{RUN_NAME}_heldout_{CV_EVAL}_pred.json", "w"))
open(f"{WORK}/labels.txt", "w").write("\n".join(LABELS) + "\n")
r = subprocess.run([sys.executable, f"{WORK}/evaluate_phase2.py", "--truth", f"{WORK}/truth_{CV_EVAL}.json",
                    "--prediction", f"{WORK}/{RUN_NAME}_heldout_{CV_EVAL}_pred.json", "--labels", f"{WORK}/labels.txt"],
                   capture_output=True, text=True)
print("official evaluator:", r.stdout.strip().splitlines()[-3] if r.returncode == 0 else r.stderr)


held-out (fold2) strict macro-F1 0.4635 (P 0.4822, R 0.4640); without boundary repair 0.2977
  APP        P=0.185 R=0.155 F1=0.168  (tp 31, fp 137, fn 169)
  RESP       P=0.240 R=0.132 F1=0.170  (tp 25, fp 79, fn 165)
  A.COUNSEL  P=0.517 R=0.480 F1=0.498  (tp 60, fp 56, fn 65)
  R.COUNSEL  P=0.521 R=0.629 F1=0.570  (tp 61, fp 56, fn 36)
  JUDGE      P=0.493 R=0.549 F1=0.519  (tp 67, fp 69, fn 55)
  WIT        P=0.405 R=0.125 F1=0.191  (tp 15, fp 22, fn 105)
  AUTH       P=0.415 R=0.299 F1=0.348  (tp 140, fp 197, fn 328)
  COURT      P=0.752 R=0.835 F1=0.791  (tp 466, fp 154, fn 92)
  STAT       P=0.629 R=0.623 F1=0.626  (tp 458, fp 270, fn 277)
  PREC       P=0.299 R=0.358 F1=0.326  (tp 200, fp 469, fn 359)
  DATE       P=0.926 R=0.917 F1=0.922  (tp 699, fp 56, fn 63)
  CASENO     P=0.404 R=0.466 F1=0.433  (tp 157, fp 232, fn 180)
errors: {'correct': 2379, 'boundary_error': 734, 'spurious': 796, 'boundary_and_label_error': 112, 'label_error': 155, 'missed_entirely': 915}
most common l

## 13. Final model, test predictions, submission zip

In [13]:
# ---- final model -> test predictions -> submission zip ----
if REFIT_ON_BOTH:
    del ner; gc.collect(); torch.cuda.empty_cache() if device.type == "cuda" else None
    ner, _ = build_ner()
    both = {**DOCS["fold1"], **DOCS["fold2"]}
    fit = train_ner(ner, both, dict(C, EPOCHS=cv["best_epoch"], PATIENCE=0), device, eval_fn=None)
    RESULTS["refit"] = {k: v for k, v in fit.items() if k != "best_state"}
    final_state = fit["best_state"]
else:
    final_state = cv["best_state"]
torch.save(final_state, f"{WORK}/{RUN_NAME}_final_state.pt")
test_pred, test_s = predict_docs(ner, TEST_DOCS, LABELS, C, device, causal, *TRIM_ALL)


def validate_prediction(pred, texts, labels):
    assert set(pred) == set(texts), f"id mismatch: missing {len(set(texts)-set(pred))}, extra {len(set(pred)-set(texts))}"
    for did, ents in pred.items():
        assert isinstance(ents, list)
        seen = set()
        for x in ents:
            s, e, l = x
            assert type(s) is int and type(e) is int and 0 <= s < e <= len(texts[did]) and l in labels, (did, x)
            assert (s, e, l) not in seen, ("duplicate", did, x); seen.add((s, e, l))


validate_prediction(test_pred, TEST_TEXTS, LABELS)
zip_path = f"{WORK}/submission_{RUN_NAME}.zip"
with open(f"{WORK}/prediction.json", "w", encoding="utf-8") as f:
    json.dump(test_pred, f)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(f"{WORK}/prediction.json", arcname="prediction.json")        # at the zip ROOT
with zipfile.ZipFile(zip_path) as z:
    assert z.namelist() == ["prediction.json"]
RESULTS["test"] = dict(docs=len(test_pred), entities=sum(len(v) for v in test_pred.values()),
                       per_label=dict(Counter(e[2] for v in test_pred.values() for e in v)), inference_seconds=round(test_s, 1),
                       model="refit on fold1+fold2" if REFIT_ON_BOTH else f"trained on {CV_TRAIN}")
print(f"wrote {zip_path}: {RESULTS['test']['entities']} entities over {len(test_pred)} documents")


Technique full_ft: trainable 190,041,369 of 190,041,369 parameters (86.901% of the Phase-1 model's 218,687,296)
training: 368 windows/epoch, 46 optimizer steps/epoch, 9 epochs max
  epoch 1: train_loss=1.53499, seconds=77.6
  epoch 2: train_loss=0.27852, seconds=154.8
  epoch 3: train_loss=0.11973, seconds=232.2
  epoch 4: train_loss=0.06222, seconds=309.4
  epoch 5: train_loss=0.03672, seconds=386.9
  epoch 6: train_loss=0.02286, seconds=464.4
  epoch 7: train_loss=0.01417, seconds=541.7
  epoch 8: train_loss=0.00887, seconds=618.9
  epoch 9: train_loss=0.00648, seconds=696.2
wrote /kaggle/working/submission_p2_12_moe_hc_full_ft_bilstm_clsw.zip: 4066 entities over 35 documents


## 14. Save everything to the bucket and print the result block

In [14]:
RESULTS["notebook_minutes"] = round((time.time() - NB_START) / 60, 1)
res_path = f"{WORK}/{RUN_NAME}_results.json"
json.dump(RESULTS, open(res_path, "w"), indent=1, default=str)
for p in (res_path, f"{WORK}/{RUN_NAME}_heldout_{CV_EVAL}_pred.json", f"{WORK}/{RUN_NAME}_cvmodel_state.pt",
          f"{WORK}/{RUN_NAME}_final_state.pt"):
    upload_model_file(p)
h, ti = RESULTS["heldout"], RESULTS["technique_info"]
print("=" * 100)
print(f"RESULT {RUN_NAME}: held-out {CV_EVAL} strict macro-F1 {h['macro_F1']} (P {h['macro_P']}, R {h['macro_R']}; "
      f"no-repair {h['macro_F1_without_boundary_repair']}) | best epoch {RESULTS['cv']['best_epoch']} | "
      f"trainable {ti['params_trainable']:,} ({ti['trainable_pct_of_backbone']}% of {PHASE1_PARAMS:,}) | "
      f"peak mem {RESULTS['cv']['peak_mem_gb']} GB | train {RESULTS['cv']['train_seconds']}s | total {RESULTS['notebook_minutes']} min")
print("per-label F1:", {l: d["F1"] for l, d in h["per_label"].items()})
print("=" * 100)
print("Submit:", f"submission_{RUN_NAME}.zip", "| send me the RESULT block above, or the file", os.path.basename(res_path))


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p2_12_moe_hc_full_ft_bilstm_clsw_results.json to bucket kavincg23/cs781-models


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

... (6 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p2_12_moe_hc_full_ft_bilstm_clsw_final_state.pt to bucket kavincg23/cs781-models
RESULT p2_12_moe_hc_full_ft_bilstm_clsw: held-out fold2 strict macro-F1 0.4635 (P 0.4822, R 0.464; no-repair 0.2977) | best epoch 9 | trainable 190,041,369 (86.901% of 218,687,296) | peak mem 5.365 GB | train 563.2s | total 22.8 min
per-label F1: {'APP': 0.1685, 'RESP': 0.1701, 'A.COUNSEL': 0.4979, 'R.COUNSEL': 0.5701, 'JUDGE': 0.5194, 'WIT': 0.1911, 'AUTH': 0.3478, 'COURT': 0.7912, 'STAT': 0.6261, 'PREC': 0.3257, 'DATE': 0.9216, 'CASENO': 0.4325}
Submit: submission_p2_12_moe_hc_full_ft_bilstm_clsw.zip | send me the RESULT block above, or the file p2_12_moe_hc_full_ft_bilstm_clsw_results.json
